In [1]:
# Task 7 & 8: Virtual Trading Execution and Performance Tracking

import yfinance as yf
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Load our allocated portfolio and predictions
print("Loading Task 5 portfolio allocations and Task 3 predictions...")
allocation = pd.read_csv('../reports/task5_final_allocation.csv')
forecasts = pd.read_csv('../data/processed/jan_2026_forecasts.csv', index_col=0)

portfolio_tickers = allocation['Stock'].tolist()

# 2. Fetch the ACTUAL market prices for our execution window
# We will use May 14 (Day 1) and May 15 (Day 2), 2026.
print("Fetching real market data for May 14 and May 15, 2026...")
# Fetching from May 13 to May 16 ensures we reliably capture the 14th and 15th
actual_data = yf.download(portfolio_tickers, start="2026-05-13", end="2026-05-16")['Close']
actual_data = actual_data.ffill().bfill()

# Extract exactly May 14 and May 15
day1_date = '2026-05-14'
day2_date = '2026-05-15'

# 3. Simulate the Trading & Compare Predictions
print("\nExecuting virtual trades and calculating P&L...")
comparison_results = []

for index, row in allocation.iterrows():
    stock = row['Stock']
    capital = row['Allocated Amount (₹)']
    
    # Get the actual market closes
    try:
        actual_day1 = actual_data.loc[day1_date, stock]
        actual_day2 = actual_data.loc[day2_date, stock]
    except KeyError:
        # Fallback if a specific date was a holiday/missing
        actual_day1 = actual_data[stock].iloc[-2]
        actual_day2 = actual_data[stock].iloc[-1]
        
    # Get our model's predictions (Pretending Jan 1 & Jan 2 are our May predictions)
    pred_day1 = forecasts.loc['2026-01-01', stock]
    pred_day2 = forecasts.loc['2026-01-02', stock]
    
    # Calculate how many shares we "bought" on Day 1
    shares_bought = capital / actual_day1
    
    # Calculate portfolio value at the end of Day 2
    final_value = shares_bought * actual_day2
    profit_loss = final_value - capital
    actual_return_pct = ((actual_day2 - actual_day1) / actual_day1) * 100
    
    comparison_results.append({
        'Stock': stock,
        'Pred Day 1': round(pred_day1, 2),
        'Actual Day 1': round(actual_day1, 2),
        'Pred Day 2': round(pred_day2, 2),
        'Actual Day 2': round(actual_day2, 2),
        'Actual Return (%)': round(actual_return_pct, 2),
        'P&L (₹)': round(profit_loss, 2)
    })

comparison_df = pd.DataFrame(comparison_results)

# 4. Calculate Total Portfolio Performance
total_initial = allocation['Allocated Amount (₹)'].sum()
total_profit = comparison_df['P&L (₹)'].sum()
total_final = total_initial + total_profit
total_return_pct = (total_profit / total_initial) * 100

# Save the final table for the report
comparison_df.to_csv('../reports/task8_final_performance.csv', index=False)

# --- Display Final Results ---
print("\n=======================================================")
print("TASK 8: PREDICTED VS ACTUAL OUTCOMES")
print("=======================================================\n")
print(comparison_df.to_string(index=False))

print("\n=======================================================")
print("FINAL PORTFOLIO PERFORMANCE SUMMARY")
print("=======================================================")
print(f"Initial Capital Deployed : ₹{total_initial:,.2f}")
print(f"Final Portfolio Value    : ₹{total_final:,.2f}")
print(f"Total Profit / Loss      : ₹{total_profit:,.2f}")
print(f"Net Portfolio Return     : {total_return_pct:.2f}%")
print("=======================================================\n")
print("Data saved to 'reports/task8_final_performance.csv'")

Loading Task 5 portfolio allocations and Task 3 predictions...
Fetching real market data for May 14 and May 15, 2026...


[*********************100%***********************]  26 of 26 completed



Executing virtual trades and calculating P&L...

TASK 8: PREDICTED VS ACTUAL OUTCOMES

        Stock  Pred Day 1  Actual Day 1  Pred Day 2  Actual Day 2  Actual Return (%)  P&L (₹)
      OFSS.NS     9265.06       8904.50     9270.54       9015.00               1.24  1352.20
   J&KBANK.NS      124.58        132.05      124.66        130.70              -1.02 -1104.66
   MANKIND.NS     2385.70       2462.20     2386.48       2503.00               1.66  1122.58
     SPARC.NS      156.88        170.08      156.85        163.65              -3.78 -2470.66
ADANIPORTS.NS     1556.16       1773.40     1557.03       1795.10               1.22   677.97
  ASIANENE.NS      309.64        306.25      309.78        313.45               2.35  1263.85
        LT.NS     3943.58       3940.40     3945.77       3909.00              -0.80  -330.99
   CARYSIL.NS      947.18        912.35      947.79        918.45               0.67   271.96
   SBILIFE.NS     1970.32       1866.70     1971.20       1864.50 